# Kaggle T4 16GB 从零部署 — Music Video Platform (严格 Dataset 模式)

**安全约束**
- HeartMuLa 3B 必须且仅 `/kaggle/input/heartmula-3b` Dataset，缺失/不完整直接报错，不回落到 `/kaggle/working`，不自动下载
- 禁止任何 HeartMuLa 权重出现在 `/kaggle/working`（6.5GB 会占满 19.5GB）
- ASR 仅 `faster-whisper-small` (~250MB int8)，禁止 `large-v3`
- GPT-SoVITS / ACE-Step 保持 Modal 远程，不在 Kaggle 下载
- HeartCodec 保持 API 模式

**顺序 A-M 必须按序执行，不跳步**


## A. 检查 Kaggle T4 GPU / CUDA

In [ ]:
!nvidia-smi
import pathlib
print(open('/proc/driver/nvidia/version').read() if pathlib.Path('/proc/driver/nvidia/version').exists() else 'no nvidia version')
print('Expect: Tesla T4 16GB, CUDA 12.4')

## B. 检查 /kaggle/working 剩余空间

In [ ]:
!df -h /kaggle/working
!du -sh /kaggle/working 2>&1 | head -n 20
import shutil
u = shutil.disk_usage('/kaggle/working')
print(f"total {u.total/1024**3:.2f}GB used {u.used/1024**3:.2f}GB free {u.free/1024**3:.2f}GB")
assert u.free/1024**3 > 2, 'WARNING: working 剩余 <2GB'

## C. 检查 /kaggle/input/heartmula-3b（严格，缺失直接报错）

In [ ]:
from backend.app.services.heartmula_service import require_heartmula_dataset, assert_no_heartmula_working_download
assert_no_heartmula_working_download()  # 禁止 working 出现权重
path = require_heartmula_dataset()  # 不存在/不完整直接抛 HeartMuLaDatasetError
print(f'OK: {path}')
!ls -lh /kaggle/input/heartmula-3b | head -n 20
!ls -lh /kaggle/input/heartmula-3b/*.safetensors | head -n 20

## D. 设置 HF_HOME / TORCH_HOME（统一缓存，避免双份）

In [ ]:
import os, pathlib
cache_root = '/kaggle/working/cache'
hf_home = f'{cache_root}/hf'
torch_home = f'{cache_root}/torch'
os.environ['KAGGLE_CACHE_ROOT'] = cache_root
os.environ['HF_HOME'] = hf_home
os.environ['HF_HUB_CACHE'] = f'{hf_home}/hub'
os.environ['TORCH_HOME'] = torch_home
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER','1')
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ.setdefault('VOICE_CLONE_ASR_MODEL','small')
os.environ.setdefault('HEARTCODEC_LOCAL_MODE','false')
os.environ.setdefault('HEARTMULA_KAGGLE_DATASET_PATH','/kaggle/input/heartmula-3b')
for p in [hf_home, torch_home, f'{cache_root}/hf', '/kaggle/working/output']:
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)
print(f"HF_HOME={hf_home}")
print(f"TORCH_HOME={torch_home}")
!bash kaggle_setup.sh  # 完整检查（含 STRICT 校验）

## E. 安装 requirements-kaggle.txt（首次执行）

In [ ]:
# 首次运行取消注释下一行；已安装可跳过
# !pip install -r requirements-kaggle.txt
print('DRY-RUN: 按用户要求本次不执行大规模 pip install，确认后取消注释执行')

## F. 检查 PyTorch CUDA

In [ ]:
import torch
print(f'torch {torch.__version__}')
print(f'cuda available {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(torch.version.cuda)

## G. 检查 faster-whisper-small 是否已缓存

In [ ]:
from backend.app.services.experimental.asr_client import is_asr_model_cached, ASR_MODEL, ASR_CACHE_DIR
import os
hf_home = os.getenv('HF_HOME', ASR_CACHE_DIR)
print(f'ASR_MODEL={ASR_MODEL} (expect small)')
print(f'cache {hf_home} cached={is_asr_model_cached("small", hf_home)}')
assert ASR_MODEL == 'small', 'ASR_MODEL must be small'

## H. 如果没有，仅允许下载 small（禁止 large-v3）

In [ ]:
# DRY-RUN：仅展示，不下载。确认后取消注释实际下载
print('DRY-RUN: 实际下载仅在首次转写时触发 WhisperModel(small, download_root=HF_HOME)，此处不批量下载')
# from faster_whisper import WhisperModel
# import os
# WhisperModel('small', device='cpu', compute_type='int8', download_root=os.getenv('HF_HOME'))

## I. 检查 HeartMuLa Dataset（严格，不允许下载）

In [ ]:
from backend.app.services.heartmula_service import require_heartmula_dataset, assert_no_heartmula_working_download
assert_no_heartmula_working_download()
path = require_heartmula_dataset()
print(f'OK: HeartMuLa STRICT OK {path} — 禁止任何下载到 /kaggle/working')

## J. 检查 R2 配置

In [ ]:
import os
for k in ['R2_ENDPOINT','R2_ACCESS_KEY_ID','R2_SECRET_ACCESS_KEY','R2_BUCKET','R2_PUBLIC_DOMAIN']:
    print(f"{k}={'SET' if os.getenv(k) else 'NOT SET'}")

## K. 检查 Modal API 配置

In [ ]:
import os
for k in ['ACE_STEP_MODAL_URL','ACE_STEP_GENERATE_URL','ACE_STEP_HEALTH_URL','HEARTMULA_API_URL','HEARTMULA_API_KEY','HEARTCODEC_API_URL','HEARTCODEC_API_KEY']:
    print(f"{k}={'SET' if os.getenv(k) else 'NOT SET'}")
print('GPT-SoVITS/ACE-Step 保持 Modal 远程，不在 Kaggle 下载')

## L. 启动 FastAPI

In [ ]:
# 后台启动
!nohup uvicorn backend.main:app --host 0.0.0.0 --port 8000 > /kaggle/working/uvicorn.log 2>&1 &
!sleep 3 && cat /kaggle/working/uvicorn.log | head -n 50

## M. 执行 health/API/AI 服务连接测试

In [ ]:
import httpx, time, os
base = os.getenv('FASTAPI_BASE_URL','http://127.0.0.1:8000')
for i in range(10):
    try:
        r = httpx.get(f'{base}/health', timeout=5)
        print(r.status_code, r.text[:500])
        break
    except Exception as e:
        print(f'retry {i}: {e}')
        time.sleep(1)
print(httpx.get(f'{base}/api/v1/services/status', timeout=5).text[:1000])